# Wild Drone Appeared — resumable Colab experiments

Use **Runtime → Change runtime type → GPU** (T4 is sufficient; A100 is optional). Run cells 1–6 in order, then cell 8 to download results. Cell 7 is optional. The notebook records the actual hardware; an A100 speedup is not assumed.

The local 200-episode validation study is not evidence of successful learning. This notebook extends the same fully observable simulator to 1,000 episodes per method and three training seeds. It changes neither reward nor battery/collision rules. Seeds 5000–5004 are validation; **do not tune on final test seeds 10000+**. More training may still fail.

This repository is private. Cell 1 asks for a GitHub token with **read access to this repository** using a hidden input. Never paste a token into code or a URL. Saved notebook cells and outputs contain no credentials. Keep results in Drive so a runtime disconnect does not erase them.


## 1. Clone the private repository

In [ ]:
import getpass
import os
from pathlib import Path
import subprocess
import sys
import tempfile

REPO = Path('/content/wild-drone-appeared')
if not (REPO / '.git').exists():
    token = getpass.getpass('GitHub token (repository read access): ')
    child_env = os.environ.copy()
    child_env.update(WILD_DRONE_GITHUB_TOKEN=token, GIT_TERMINAL_PROMPT='0')
    try:
        with tempfile.TemporaryDirectory() as temporary:
            helper = Path(temporary) / 'askpass.py'
            helper.write_text("#!/usr/bin/env python3\nimport os, sys\n"
                              "print('x-access-token' if 'Username' in sys.argv[1] "
                              "else os.environ['WILD_DRONE_GITHUB_TOKEN'])\n")
            helper.chmod(0o700)
            result = subprocess.run(
                ['git', '-c', 'credential.helper=', '-c', f'core.askPass={helper}', 'clone',
                 'https://github.com/SayuruRehan/wild-drone-appeared.git', str(REPO)],
                env=child_env, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            if result.returncode:
                raise RuntimeError('Clone failed. Check token repository access, then rerun cell 1.')
    finally:
        token = ''
        child_env.pop('WILD_DRONE_GITHUB_TOKEN', None)
os.chdir(REPO)
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Install and verify
Colab already supplies its GPU PyTorch build. Keep it; do not install the CPU wheel here.
The install cell also activates the repository source in the running notebook kernel and verifies a direct package import; no runtime restart is required.


In [ ]:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[rl,dev]', 'pandas>=2,<3'], check=True)
# Editable installation affects new Python processes; activate src in this live kernel too.
import importlib
SOURCE = (REPO / 'src').resolve()
if not (SOURCE / 'wild_drone' / '__init__.py').is_file():
    raise RuntimeError('Repository source is missing. Rerun section 1 before installing.')
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
importlib.invalidate_caches()
import wild_drone
from wild_drone.config import EnvConfig
print('Notebook package:', wild_drone.__file__)
subprocess.run([sys.executable, '-m', 'ruff', 'check', 'src', 'tests', 'scripts'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('No CUDA runtime: benchmark and experiment will use CPU. Select a GPU runtime to test CUDA.')


## 3. Bounded CPU/GPU benchmark
Five episodes per method/device. Compare steps per wall second; the NumPy simulator still runs on CPU. These short runs diagnose throughput, not learning. Choose CPU below if it is faster.

In [ ]:
import json
import time
import pandas as pd
from wild_drone.config import EnvConfig
from wild_drone.learning import TrainConfig, train

raw = json.loads((REPO / 'configs/development.json').read_text())
benchmark_rows = []
with tempfile.TemporaryDirectory() as temporary:
    for device in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
        for algorithm in ['dqn', 'reinforce', 'a2c']:
            settings = TrainConfig(**(raw['training'] | {
                'episodes': 5, 'device': device, 'progress_interval': 0}))
            started = time.perf_counter()
            report = train(algorithm, EnvConfig(**raw['environment']), settings,
                           Path(temporary) / device / algorithm)
            seconds = time.perf_counter() - started
            benchmark_rows.append(dict(algorithm=algorithm, device=device,
                                       seconds=seconds, steps=report['steps'],
                                       steps_per_second=report['steps']/seconds))
benchmark = pd.DataFrame(benchmark_rows)
display(benchmark)


## 4. Mount Drive and set the experiment budget
Use the same folder to resume. `EPISODES` means the total budget, not extra episodes. A new runtime must rerun cells 1–4. Changing learning or environment settings requires a new run folder; resume rejects such changes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/wild-drone-appeared')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Choose CPU if benchmark is faster.
EPISODES = 1000
TRAINING_SEEDS = [0, 1, 2]
VALIDATION_SEEDS = [5000, 5001, 5002, 5003, 5004]
PILOT = RUN_ROOT / 'validation20-v2'
benchmark.to_csv(RUN_ROOT / 'runtime-benchmark.csv', index=False)


## 5. Run or resume the 20×20 validation study

This trains DQN, Monte Carlo REINFORCE with a learned baseline, and n-step A2C for each training seed, then evaluates both final and best-validation checkpoints against four conventional controllers. Logs, CSVs and full states are stored in Drive. Every 20 completed episodes is a durable boundary; a disconnect can lose work since that boundary. Progress appears below every 20 episodes. If you stop the cell, it stops and waits for the training process before returning; rerun this cell to resume from the last saved boundary. Do not run two sessions into the same folder.

The best model uses maximum mean validation return; its validation score is selection-biased and is not a final generalisation result. Scenario rows from one fitted model are not independent training runs.


In [ ]:
from wild_drone.notebook_runtime import run_with_progress

command = [sys.executable, '-u', 'scripts/run_validation_pilot.py',
           '--config', 'configs/development.json', '--episodes', str(EPISODES),
           '--training-seeds', *map(str, TRAINING_SEEDS),
           '--validation-seeds', *map(str, VALIDATION_SEEDS),
           '--device', DEVICE, '--out', str(PILOT)]
run_with_progress(command)


## 6. Inspect learning and failure modes
Inspect return alongside freshness, drone losses and collisions. A stationary or crashing policy can have misleading individual metrics. This table averages scenarios within each fitted seed before aggregating fitted seeds.

In [ ]:
from IPython.display import Image, display
results = pd.read_csv(PILOT / 'validation_comparison.csv')
metrics = ['return', 'weighted_freshness', 'overdue_fraction', 'total_losses',
           'collision_attempts', 'safe_returns']
rl = results[results.training_seed.notna()]
per_fit = rl.groupby(['policy', 'training_seed'])[metrics].mean()
display(per_fit.groupby('policy').agg(['mean', 'std']))
display(results[results.training_seed.isna()].groupby('policy')[metrics].mean())
for algorithm in ['dqn', 'reinforce', 'a2c']:
    display(Image(filename=str(PILOT / f'{algorithm}-learning.png')))


## 7. Optional 40×40 study

Leave disabled until the 20×20 results have been assessed. This is a separately trained environment, not an out-of-distribution transfer claim. It uses the main config and a separate Drive folder. Grid, episode and scenario budgets should be chosen on validation before final testing. Reward/constraint ablations and calibrated fire dynamics remain later work.


In [ ]:
RUN_40 = False
if RUN_40:
    run_with_progress([sys.executable, '-u', 'scripts/run_validation_pilot.py',
                    '--config', 'configs/main.json', '--episodes', str(EPISODES),
                    '--training-seeds', *map(str, TRAINING_SEEDS),
                    '--validation-seeds', *map(str, VALIDATION_SEEDS),
                    '--device', DEVICE, '--out', str(RUN_ROOT / 'validation40-v2')])


## 8. Download the results for analysis

Return **wild-drone-results.zip** in the chat after cell 5 finishes (or after interruption if you need help). It includes protocol, provenance, CSVs and figures. Model/replay checkpoints remain in Drive; they can be large and are excluded by default. Set `INCLUDE_CHECKPOINTS=True` only when checkpoint inspection is needed. The archive contains no token or repository credentials.

Next: analyse repeated learning/failure curves, select validation-only changes and ablations, freeze the protocol, then run a separate final comparison on unseen test scenarios. The notebook deliberately does not consume final tests during tuning.


In [ ]:
from google.colab import files
import zipfile

from wild_drone.checkpoint_audit import audit_checkpoints
from wild_drone.training_io import atomic_json_save

# Recheck Drive exports immediately before packaging. A stale export stops the archive.
for state in sorted(RUN_ROOT.rglob('training_state.pt')):
    report = audit_checkpoints(state.parent)
    atomic_json_save(report, state.parent/'checkpoint_manifest.json')
    print(f'Verified {state.parent}: {report["completed_episodes"]} episodes', flush=True)

INCLUDE_CHECKPOINTS = False
archive = Path('/content/wild-drone-results.zip')
allowed = {'.csv', '.json', '.png'} | ({'.pt'} if INCLUDE_CHECKPOINTS else set())
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN_ROOT.rglob('*'):
        if path.is_file() and path.suffix in allowed:
            z.write(path, str(path.relative_to(RUN_ROOT)))
files.download(str(archive))
